# Lesson 2: Thresholding, Erosion, and Dilation

Thresholding converts a grayscale image into a binary (black/white) image by comparing each pixel to a cutoff value. The result is often noisy, so we clean it up with two basic morphological operations: **erosion** (shrinks white regions, removes small specks) and **dilation** (grows white regions, fills small holes).

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

## Build a noisy test image

We synthesize a grayscale image with a bright shape on a dark background, then add random noise so some background pixels are bright and some foreground pixels are dark.

In [ ]:
rng = np.random.default_rng(0)

gray = np.full((120, 120), 40, dtype=np.uint8)
cv2.circle(gray, (60, 60), 35, 220, -1)

noise = rng.normal(0, 35, gray.shape)
noisy = np.clip(gray.astype(np.int16) + noise, 0, 255).astype(np.uint8)

# sprinkle a few salt-and-pepper specks
speckle_coords = rng.integers(0, 120, size=(60, 2))
for y, x in speckle_coords:
    noisy[y, x] = 255 if noisy[y, x] < 128 else 0

plt.imshow(noisy, cmap='gray', vmin=0, vmax=255)
plt.title('Noisy grayscale image')
plt.axis('off')
plt.show()

## Thresholding

`cv2.threshold` sets every pixel above the cutoff to 255 and every pixel at or below it to 0.

In [ ]:
threshold_value = 128
_, binary = cv2.threshold(noisy, threshold_value, 255, cv2.THRESH_BINARY)

plt.imshow(binary, cmap='gray', vmin=0, vmax=255)
plt.title(f'Thresholded (t={threshold_value})')
plt.axis('off')
plt.show()

Notice the salt-and-pepper specks and ragged edges left over from the noise. This is exactly what morphological operations clean up.

## Erosion

Erosion slides a small structuring element (kernel) over the image; a pixel stays white only if the *entire* kernel fits inside the white region. This shrinks white regions and removes small white specks.

In [ ]:
kernel = np.ones((3, 3), np.uint8)
eroded = cv2.erode(binary, kernel, iterations=1)

plt.imshow(eroded, cmap='gray', vmin=0, vmax=255)
plt.title('Eroded')
plt.axis('off')
plt.show()

## Dilation

Dilation does the opposite: a pixel becomes white if the kernel overlaps the white region *at all*. This grows white regions and fills small black holes/specks.

In [ ]:
dilated = cv2.dilate(binary, kernel, iterations=1)

plt.imshow(dilated, cmap='gray', vmin=0, vmax=255)
plt.title('Dilated')
plt.axis('off')
plt.show()

## Opening: erosion followed by dilation

Applying erosion then dilation (an **opening**) removes small white specks while restoring the size of the main region — a common way to denoise a thresholded image.

In [ ]:
opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, img, title in zip(
    axes,
    [binary, eroded, dilated, opened],
    ['Thresholded', 'Eroded', 'Dilated', 'Opened\n(erode then dilate)'],
):
    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    ax.set_title(title, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

### Exercise

1. Try `cv2.MORPH_CLOSE` (dilation followed by erosion) instead of `MORPH_OPEN`. How does it differ, and which black-pixel noise does it fix that opening does not?
2. Increase the kernel size to `(5, 5)`. How does that change the result compared to `(3, 3)`?